# Pattern 2: Access Control with Metadata Filters

Use metadata sidecar files to tag documents with access control attributes (department,
role, user ID), then apply filters at retrieval time so users only see documents they're
authorized to access.

This is the same approach described in the AWS blog post
[Access control for vector stores using metadata filtering](https://aws.amazon.com/blogs/machine-learning/access-control-for-vector-stores-using-metadata-filtering-with-knowledge-bases-for-amazon-bedrock/),
adapted for Managed Knowledge Bases.

**What you get:** Document-level access control without changing the KB or data source.

**What you don't get:** Per-user identity enforcement — the calling application decides
which filters to apply. There's no server-side enforcement layer (that's Pattern 3+).

## How It Works

```
1. Tag documents with access metadata:
   octank_financial_10K.pdf.metadata.json → {"department": "finance", "access_level": "confidential"}

2. At retrieval time, app injects filter based on authenticated user:
   User (finance) ──► Retrieve API + {department: "finance"} ──► Managed KB
                                                                  │
                       Only finance docs returned ◄────────────────┘
```

## Use Cases

- **Multi-department chatbot**: HR docs only visible to HR, finance docs to finance
- **Multi-tenant SaaS**: Each tenant's documents tagged with `tenant_id`, filtered per request
- **Healthcare**: Patient transcripts tagged with `patient_id`, doctors only see their patients
- **Compliance**: Restrict sensitive documents by `access_level` / `classification`

## Prerequisites

- Run [Pattern 1](01-direct-sdk.ipynb) first — it creates the shared bucket, uploads the
  sample documents **with metadata sidecars**, and creates the KB execution role.
  (Running the setup cell here again is safe — it's idempotent.)
- IAM permissions for Bedrock, S3, and IAM role creation.

In [ ]:
import boto3
import time
import json
import util   # util.py in this folder — shared bucket + upload + role setup

# --- Configuration ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"
S3_PREFIX = "documents/"

session = boto3.Session()

# Reuse the SAME bucket + docs + role as Pattern 1. setup() is idempotent, so
# if you already ran Pattern 1 this just returns the existing values (nothing
# is re-created). The docs are uploaded WITH department / access_level metadata
# sidecars (util.SAMPLE_FILE_METADATA) — that's what the filters below match on.

ROLE_ARN = None

if ROLE_ARN == None:
    info = util.setup(
        bucket_name=S3_BUCKET,
        prefix=S3_PREFIX,
        metadata=util.SAMPLE_FILE_METADATA,
        region_name=REGION,
    )

    ROLE_ARN  = info["role_arn"]
    S3_BUCKET = info["bucket"]
    S3_PREFIX = info["prefix"]

cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)
S3_ACCOUNT = session.client("sts").get_caller_identity()["Account"]

print(f"boto3 {boto3.__version__}")
print(f"Bucket: {S3_BUCKET}  Prefix: {S3_PREFIX}")
print(f"Metadata tags: {json.dumps(util.SAMPLE_FILE_METADATA, indent=2)}")


Bucket already exists: ytayeb-test02-my-bmkb-bucket
Uploading 2 file(s) to s3://ytayeb-test02-my-bmkb-bucket/documents/
  Uploaded: documents/octank_financial_10K.pdf
    + documents/octank_financial_10K.pdf.metadata.json
  Uploaded: documents/tornadoes_report.pdf
    + documents/tornadoes_report.pdf.metadata.json
  Done (2 uploaded).
Role already exists: AmazonBedrockExecutionRoleForKB_us-west-2-004970146111
  Policy already exists: AmazonBedrockFoundationModelPolicyForKnowledgeBase_us-west-2-004970146111
  Policy already exists: AmazonBedrockCloudWatchPolicyForKnowledgeBase_us-west-2-004970146111
  Policy already exists: AmazonBedrockS3PolicyForKnowledgeBase_us-west-2-004970146111
  Waiting 10s for IAM propagation...
ROLE_ARN = arn:aws:iam::004970146111:role/AmazonBedrockExecutionRoleForKB_us-west-2-004970146111
boto3 1.43.39
Bucket: ytayeb-test02-my-bmkb-bucket  Prefix: documents/
Metadata tags: {
  "octank_financial_10K.pdf": {
    "department": "finance",
    "access_level": "conf

## Step 1: Metadata Sidecar Files

For each document in S3, there is a `.metadata.json` sidecar in the same prefix.
The metadata attributes are indexed during ingestion and available for filtering.
`util.setup(..., metadata=util.SAMPLE_FILE_METADATA)` (run in Pattern 1 and again
here, idempotently) creates these sidecars for the sample documents.

### The sample documents and their tags

```
s3://<bucket>/documents/
  ├── octank_financial_10K.pdf
  ├── octank_financial_10K.pdf.metadata.json   → department=finance,    access_level=confidential
  ├── tornadoes_report.pdf
  └── tornadoes_report.pdf.metadata.json        → department=operations, access_level=internal
```

Each sidecar uses the typed `metadataAttributes` format Bedrock expects:

```json
{
  "metadataAttributes": {
    "department": {
      "value": { "type": "STRING", "stringValue": "finance" },
      "includeForEmbedding": true
    },
    "access_level": {
      "value": { "type": "STRING", "stringValue": "confidential" },
      "includeForEmbedding": true
    }
  }
}
```

The filename convention is strict: `<document-name>.metadata.json` in the same
S3 prefix as the document.


In [6]:
# Step 2: Create KB + Data Source + Ingest (same as Pattern 1)
response = cp.create_knowledge_base(
    name=f"p2-access-control-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)
kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB: {kb_id}")

for _ in range(30):
    if cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"] == "ACTIVE":
        break
    time.sleep(5)
print("KB ACTIVE")

response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {
                    "inclusionPrefixes": [S3_PREFIX]
                },
                "deletionProtectionConfiguration": {
                    "enableDeletionProtection": False
                }
            },
            "deletionProtectionConfiguration": {
                "deletionProtectionStatus": "DISABLED"
            }
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)
ds_id = response["dataSource"]["dataSourceId"]
print(f"DS: {ds_id}")

for _ in range(12):
    if cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"] == "AVAILABLE":
        break
    time.sleep(5)
print("DS AVAILABLE")

# Ingest
response = cp.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = response["ingestionJob"]["ingestionJobId"]
print(f"Ingestion job: {job_id}")

for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)

stats = job.get("statistics", {})
print(f"Ingestion {job['status']} — scanned={stats.get('numberOfDocumentsScanned', 0)}, indexed={stats.get('numberOfNewDocumentsIndexed', 0)}")

KB: FIFIJWCIE2
KB ACTIVE
DS: IECSBKF7FD
DS AVAILABLE
Ingestion job: QZKIJ8AMQX
Ingestion COMPLETE — scanned=2, indexed=2


In [7]:
# Helper to display results
def show_results(results):
    for i, res in enumerate(results, 1):
        score = res["score"]
        text = res["content"]["text"][:80]
        meta = {k: v for k, v in res.get("metadata", {}).items() if not k.startswith("_")}
        src = res.get("location", {}).get("s3Location", {}).get("uri", "").split("/")[-1]
        print(f"  {i}. score={score:.4f} | {src} | meta={json.dumps(meta)}")
    print(f"  Total: {len(results)} results")

## Step 3: Access Control Queries

In a real application, the filter values come from the authenticated user's identity
(e.g., Cognito JWT claims, SSO attributes, or a database lookup). The application
injects the filter — the user never controls it directly.

In [8]:
# Scenario: No filter (admin view — sees everything)
print("Admin view (no filter):")
response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "What regulatory compliance risks does the company face?"},
    retrievalConfiguration={
        "managedSearchConfiguration": {"numberOfResults": 5}
    }
)
show_results(response.get("retrievalResults", []))

Admin view (no filter):
  1. score=0.6903 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  2. score=0.6810 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  3. score=0.6565 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  4. score=0.5968 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  5. score=0.5554 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  Total: 5 results


In [9]:
# Scenario: Finance user — only sees finance department docs
# In production, "finance" comes from the user's JWT claim or SSO attribute
authenticated_department = "finance"

print(f"Finance user (department={authenticated_department}):")
response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "What regulatory compliance risks does the company face?"},
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "equals": {"key": "department", "value": authenticated_department}
            }
        }
    }
)
show_results(response.get("retrievalResults", []))

Finance user (department=finance):
  1. score=0.6903 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  2. score=0.6810 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  3. score=0.6565 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  4. score=0.5968 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  5. score=0.5554 | octank_financial_10K.pdf | meta={"access_level": "confidential", "department": "finance"}
  Total: 5 results


In [10]:
# Scenario: Operations user — only sees operations department docs
authenticated_department = "operations"

print(f"Operations user (department={authenticated_department}):")
response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "Where can we expect tornadoes?"},
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "equals": {"key": "department", "value": authenticated_department}
            }
        }
    }
)
show_results(response.get("retrievalResults", []))

Operations user (department=operations):
  1. score=0.5556 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  2. score=0.5010 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  3. score=0.2699 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  4. score=0.2684 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  5. score=0.2468 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  Total: 5 results


In [11]:
# Scenario: Cross-department user — sees finance AND operations docs
# Use IN filter when a user has access to multiple departments
user_departments = ["finance", "operations"]

print(f"Cross-department user (departments={user_departments}):")
response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "Where can we expect tornadoes?"},
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "in": {"key": "department", "value": user_departments}
            }
        }
    }
)
show_results(response.get("retrievalResults", []))

Cross-department user (departments=['finance', 'operations']):
  1. score=0.7094 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  2. score=0.6812 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  3. score=0.5739 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  4. score=0.5628 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  5. score=0.5545 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  Total: 5 results


In [12]:
# Scenario: Compound filter — department + access level
# Operations user who only has "internal" clearance
# (the tornadoes report is department=operations, access_level=internal → matches;
#  the finance 10-K is access_level=confidential → excluded)
print("Operations + internal access level only:")
response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "Where can we expect tornadoes? give me an overview."},
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "andAll": [
                    {"equals": {"key": "department", "value": "operations"}},
                    {"equals": {"key": "access_level", "value": "internal"}}
                ]
            }
        }
    }
)
show_results(response.get("retrievalResults", []))

Operations + internal access level only:
  1. score=0.5405 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  2. score=0.4649 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  3. score=0.2917 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  4. score=0.2463 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  5. score=0.2431 | tornadoes_report.pdf | meta={"access_level": "internal", "department": "operations"}
  Total: 5 results


## Step 4: Filtered RAG — retrieve + generate a full answer

Metadata filtering isn't just for narrowing search results — it enforces access
control on the **generated answer** too. Here we retrieve *only* the finance user's
documents, then pass those chunks to the model (Converse API) to synthesize an
answer. The model can only ground its response in documents the filter allowed.

In [ ]:
# --- Filtered RAG: retrieve (finance only) + generate with Converse ---
# Paste a generation model ARN (inference-profile ARN for CRIS-only models like Claude 4.x)
# GENERATION_MODEL = "arn:aws:bedrock:us-west-2:<ACCOUNT-NUMBER>:inference-profile/us.anthropic.claude-sonnet-4-6"
GENERATION_MODEL = "<add-generation model arn like the example above>"

QUESTION = "What are the company's key financial results and risks?"
authenticated_department = "finance"

# 1. Retrieve ONLY the documents this user is allowed to see
response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": QUESTION},
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "equals": {"key": "department", "value": authenticated_department}
            }
        }
    }
)
chunks = response.get("retrievalResults", [])
print(f"Retrieved {len(chunks)} chunk(s) for department={authenticated_department}")

# 2. Build a grounded prompt from the filtered chunks
context = "\n\n".join(
    f"[{i}] {res['content']['text']}" for i, res in enumerate(chunks, 1)
)
prompt = (
    "Answer the question using only the context below. "
    "Cite sources by their [number]. If the answer isn't in the context, say so.\n\n"
    f"Context:\n{context}\n\n"
    f"Question: {QUESTION}"
)

# 3. Generate the answer via the Converse API
brt = session.client("bedrock-runtime", region_name=REGION)
gen = brt.converse(
    modelId=GENERATION_MODEL,
    messages=[{"role": "user", "content": [{"text": prompt}]}],
    inferenceConfig={"maxTokens": 1000, "temperature": 0.0},
)

answer = gen["output"]["message"]["content"][0]["text"]
print("\n=== Generated Answer (finance-scoped) ===")
print(answer)
print(f"\n(grounded in {len(chunks)} filtered chunk(s) — the model never saw other departments' docs)")


Retrieved 5 chunk(s) for department=finance

=== Generated Answer (finance-scoped) ===
## Octank Financial: Key Financial Results and Risks

### Key Risks

Based on the provided context, Octank Financial faces several significant risks:

#### Economic & Industry Risks
- Subject to changes in **interest rates, inflation, and global economic conditions**
- A downturn in the economy could negatively impact revenue and profitability [1][4]

#### Personnel & Operational Risks
- Heavy **dependence on key personnel**, such as PersonA (CFO), whose departure could materially impact financial performance [4]
- **Dependence on key suppliers** for critical components; loss of suppliers due to bankruptcy or supply chain disruptions could be harmful [3][5]

#### Regulatory & Compliance Risks
- Operating in a **highly regulated industry**, where non-compliance could result in fines, penalties, and reputational damage [4]

#### Cybersecurity Risks
- Vulnerable to **cyber attacks** that could result in

In [ ]:
# Cleanup
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted KB {kb_id} and data source {ds_id}")

## Integration Pattern

In production, the filter injection flow looks like this:

```
1. User authenticates (Cognito / SSO / custom auth)
2. App reads user attributes (department, role, tenant_id)
3. App builds metadata filter from user attributes
4. App calls Retrieve API with filter injected
5. KB returns only documents matching the filter
```

Example with Cognito:
```python
# After Cognito authentication
user_dept = jwt_claims["custom:department"]  # from ID token

response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": user_query},
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "equals": {"key": "department", "value": user_dept}
            }
        }
    }
)
```

## Supported Filter Types

| Filter | Syntax | Access Control Example |
|---|---|---|
| Equals | `{"equals": {"key": "k", "value": "v"}}` | `department = "engineering"` |
| Not Equals | `{"notEquals": {"key": "k", "value": "v"}}` | `classification != "top-secret"` |
| In | `{"in": {"key": "k", "value": ["a","b"]}}` | `tenant_id in ["acme", "globex"]` |
| AND | `{"andAll": [filter1, filter2]}` | `department = "eng" AND level = "internal"` |
| OR | `{"orAll": [filter1, filter2]}` | `department = "eng" OR department = "security"` |
| Greater Than | `{"greaterThan": {"key": "k", "value": n}}` | `clearance_level > 3` |
| Less Than | `{"lessThan": {"key": "k", "value": n}}` | `sensitivity < 5` |

## Security Considerations

| Strength | Limitation |
|---|---|
| No KB or data source changes needed | Caller must know the KB ID |
| Works with any authentication system | App must enforce filter injection — no server-side enforcement |
| Flexible multi-attribute filtering | A malicious caller can omit filters and see everything |
| Low latency — filtering happens in the vector store | Metadata must be maintained alongside documents |

**Key risk:** The application layer is responsible for injecting the correct filter.
If a caller bypasses the app and calls the API directly without filters, they see all documents.

**Next:** [Pattern 3](03-gateway-iam.ipynb) adds a Gateway to hide the KB ID and centralize access,
making it harder to bypass the application layer.